# Data Generator — FarmIA Multi-Domain

Notebook parametrizado universal que genera datos sintéticos para cualquier dominio del Data Lakehouse.

**Arquitectura:**
- **IoT** → Genera datos + Publica en **Azure Event Hubs** (topic `sensors`, formato Avro)
- **Otros dominios** (Weather, Sales, Inventory) → Genera datos + Escribe en **Blob Storage** (landing path, formato del dominio)

**Flujo:**
1. Seleccionar dominio (widget dropdown)
2. Cargar contrato YAML → extraer schema, path, formato, opciones
3. Generar N registros sintéticos con corrupción simulada
4. **Si IoT:** Serializar a Avro → Publicar a Event Hubs
5. **Si otros:** Escribir a Blob Storage (landing) en formato del YAML
6. Monitorear: registros procesados, éxito/fallo, latencia

In [0]:
%pip install azure-eventhub azure-identity fastavro tenacity

  Using cached azure_eventhub-5.15.1-py3-none-any.whl.metadata (30 kB)
  Using cached fastavro-1.12.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.8 kB)
Using cached azure_eventhub-5.15.1-py3-none-any.whl (317 kB)
Using cached fastavro-1.12.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.4 MB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Celda 1: Importar librerías y configurar credenciales

import yaml
import json
import random
import uuid
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, 
    IntegerType, TimestampType, DateType
)

# Azure Event Hubs
from azure.eventhub import EventHubProducerClient, EventData
from azure.identity import DefaultAzureCredential

import fastavro
from io import BytesIO

from tenacity import retry, stop_after_attempt, wait_exponential

# Blob storage
STORAGE_ACCOUNT_NAME = "stfarmia"
SECRET_SCOPE_NAME = "scope-farmia"
SECRET_KEY_NAME = "kv-farmia-core"

# Event Hubs
SECRET_EH_SCOPE_NAME = "scope-eh-connection-farmia"
SECRET_EH_CONNECTION_NAME = "kv-eh-connection-farmia"
SECRET_EH_PRIMARY_KEY_NAME = "kv-eh-primary-key-farmia"

# Databricks busca encriptadamente en Azure Key Vault en tiempo de ejecución
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=SECRET_KEY_NAME)

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", 
    STORAGE_ACCOUNT_KEY
)

print("Librerías importadas exitosamente")

Librerías importadas exitosamente


In [0]:
# Celda 1: Widgets de configuración universal

import time

dbutils.widgets.dropdown("domain", "iot_sensors", ["iot_sensors", "weather_external", "sales_online", "inventory_erp"], "Dominio")
dbutils.widgets.text("num_records", "1000", "Registros por lote")
dbutils.widgets.text("corrupt_pct", "5", "% datos corruptos")
dbutils.widgets.text("min_duration", "5", "Duración ejecución (minutos)")

# Widgets opcionales (solo para IoT + Event Hubs)
dbutils.widgets.text("eh_namespace", "stfarmia-ns", "[IoT] Namespace Event Hubs")
dbutils.widgets.text("eh_name", "telemetry", "[IoT] Nombre Event Hub")
dbutils.widgets.text("batch_size", "100", "[IoT] Tamaño batch Event Hubs")

domain = dbutils.widgets.get("domain")
num_records_per_batch = int(dbutils.widgets.get("num_records"))
corrupt_pct = float(dbutils.widgets.get("corrupt_pct")) / 100.0
min_duration = int(dbutils.widgets.get("min_duration"))

eh_namespace = dbutils.widgets.get("eh_namespace")
eh_name = dbutils.widgets.get("eh_name")
batch_size = int(dbutils.widgets.get("batch_size"))

print(f"Configuración:")
print(f"  Dominio: {domain}")
print(f"  Registros por lote: {num_records_per_batch}")
print(f"  % Corruptos: {corrupt_pct * 100:.1f}%")
print(f"  Duración máxima: {min_duration} minutos")

if domain == "iot_sensors":
    print(f"  Event Hubs: {eh_namespace}/{eh_name}")
    print(f"  Batch size: {batch_size}")
else:
    print(f"  Destino: Blob Storage landing")

print(f"\nIniciando generación continua durante {min_duration} minutos...")

Configuración:
  Dominio: iot_sensors
  Registros: 1000
  % Corruptos: 5.0%
  Event Hubs: stfarmia-ns/telemetry
  Batch size: 100


In [0]:
# Celda 2: Cargar contrato YAML del dominio

DOMAIN_YAML_MAP = {
    "iot_sensors": "iot_domain.yaml",
    "weather_external": "weather_domain.yaml",
    "sales_online": "sales_domain.yaml",
    "inventory_erp": "inventory_domain.yaml",
}

CONTROL_PLANE_PATH = Path("/Workspace/Users/cbarro05@ucm.es/farmia-data-lakehouse/specs/control_plane")

yaml_file = CONTROL_PLANE_PATH / DOMAIN_YAML_MAP[domain]

with open(yaml_file) as f:
    contract = yaml.safe_load(f)

# Extraer configuración común
schema_fields = contract["schema_validation"]["fields"]
rescue_column = contract["schema_validation"]["rescued_data_column"]
source_path = contract["source"]["path"]
source_format = contract["source"]["format"]
sink_mode = contract.get("sink", {}).get("mode", "append")
sink_table = contract.get("sink", {}).get("table_name", f"{domain}_table")
partition_by = contract.get("sink", {}).get("partition_by", [])

print(f"Contrato YAML cargado: {contract['pipeline_info']['domain']}")
print(f"  Schema: {[f['name'] for f in schema_fields]}")
print(f"  Formato: {source_format}")
print(f"  Sink mode: {sink_mode}")
print(f"  Particiones: {partition_by if partition_by else 'Sin partición'}")

Contrato YAML cargado: iot_sensors
  Schema: ['sensor_id', 'location_id', 'temperature', 'humidity', 'ph', 'soil_moisture', 'reading_timestamp', 'reading_date']
  Formato: avro
  Sink mode: append
  Particiones: ['sensor_id', 'reading_date']


In [0]:
# Celda 3: Configurar conexión (condicional: IoT → Event Hubs, otros → Spark)

producer_client = None


if domain == "iot_sensors":
    # Configurar Event Hubs para IoT
    try:
        connection_str = dbutils.secrets.get(scope=SECRET_EH_SCOPE_NAME, key=SECRET_EH_CONNECTION_NAME)
        primary_key = dbutils.secrets.get(scope=SECRET_EH_SCOPE_NAME, key=SECRET_EH_PRIMARY_KEY_NAME)
    except:
        endpoint_suffix = "servicebus.windows.net"
        connection_str = f"Endpoint=sb://{eh_namespace}.{endpoint_suffix}/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey={primary_key}"
    
    try:
        from azure.eventhub import EventHubProducerClient
        producer_client = EventHubProducerClient.from_connection_string(
            connection_str,
            eventhub_name=eh_name,
            logging_enable=False
        )
        with producer_client:
            pass
        print(f"Conectado a Event Hub: {eh_namespace}/{eh_name}")
    except Exception as e:
        print(f"Error de conexión Event Hubs: {e}")
else:
    # Para otros dominios, se escribe directamente a Blob Storage via Spark
    print(f"Configurado para escribir en Blob Storage: {source_path}")

Conectado a Event Hub: stfarmia-ns/telemetry


In [0]:
# Celda 4: Generar datos sintéticos (universal para todos los dominios)

# Configuración de rangos realistas por dominio/campo
DOMAIN_FIELD_CONFIG = {
    "iot_sensors": {
        "sensor_id": {"prefix": "SENSOR", "range": (1, 50)},
        "location_id": {"prefix": "LOC", "range": (1, 20)},
        "temperature": {"min": 15.0, "max": 40.0},
        "humidity": {"min": 30.0, "max": 95.0},
        "ph": {"min": 4.0, "max": 9.0},
        "soil_moisture": {"min": 10.0, "max": 80.0},
        "reading_timestamp": {"days_back": 7},
        "reading_date": {"from_timestamp": "reading_timestamp"},
    },
    "weather_external": {
        "city": {"values": ["Madrid", "Barcelona", "Sevilla", "Valencia", "Bilbao", "Zaragoza", "Málaga", "Murcia", "Palma", "Granada"]},
        "temperature": {"min": -5.0, "max": 45.0},
        "humidity": {"min": 20.0, "max": 100.0},
        "forecast_timestamp": {"days_back": 7},
        "forecast_date": {"from_timestamp": "forecast_timestamp"},
    },
    "sales_online": {
        "transaction_id": {"uuid": True},
        "customer_id": {"prefix": "CUST", "range": (1, 10000)},
        "product_id": {"prefix": "PROD", "range": (1, 500)},
        "amount": {"min": 1.0, "max": 500.0},
        "quantity": {"min": 1, "max": 20},
        "currency": {"values": ["EUR"]},
        "transaction_timestamp": {"days_back": 30},
        "transaction_date": {"from_timestamp": "transaction_timestamp"},
    },
    "inventory_erp": {
        "product_id": {"prefix": "PROD", "range": (1, 500)},
        "warehouse_id": {"prefix": "WH", "range": (1, 20)},
        "quantity_available": {"min": 0, "max": 10000},
        "quantity_reserved": {"min": 0, "max": 500},
        "last_updated_timestamp": {"days_back": 7},
        "last_updated": {"from_timestamp": "last_updated_timestamp"},
        "operation_id": {"uuid": True},
    },
}

CORRUPTION_TYPES = ["null_required", "out_of_range", "wrong_format", "missing_field", "late_timestamp"]

TYPE_MAP = {
    "StringType": StringType(),
    "DoubleType": DoubleType(),
    "IntegerType": IntegerType(),
    "TimestampType": TimestampType(),
    "DateType": DateType(),
}

def generate_value(field_name: str, field_type: str, config: dict, row: dict = None) -> Any:
    """Generar valor sintético para un campo."""
    field_config = config.get(field_name, {})
    now = datetime.now()

    if "uuid" in field_config and field_config["uuid"]:
        return str(uuid.uuid4())

    if "values" in field_config:
        return random.choice(field_config["values"])

    if "prefix" in field_config:
        prefix = field_config["prefix"]
        low, high = field_config["range"]
        return f"{prefix}-{random.randint(low, high):04d}"

    if "from_timestamp" in field_config:
        source_col = field_config["from_timestamp"]
        source_val = row.get(source_col) if row else None
        if source_val and isinstance(source_val, datetime):
            return source_val.date()
        return None

    if field_type == "StringType":
        return f"{field_name}_{random.randint(1, 9999):04d}"
    elif field_type == "DoubleType":
        min_val = field_config.get("min", 0.0)
        max_val = field_config.get("max", 100.0)
        return round(random.uniform(min_val, max_val), 2)
    elif field_type == "IntegerType":
        min_val = field_config.get("min", 0)
        max_val = field_config.get("max", 1000)
        return random.randint(min_val, max_val)
    elif field_type == "TimestampType":
        days_back = field_config.get("days_back", 7)
        random_seconds = random.randint(0, days_back * 86400)
        ts = now - timedelta(seconds=random_seconds)
        return ts
    elif field_type == "DateType":
        days_back = field_config.get("days_back", 7)
        return (now - timedelta(days=random.randint(0, days_back))).date()
    
    return None

def corrupt_value(field_name: str, field_type: str, nullable: bool) -> Any:
    """Generar valor corrupto simulado."""
    corruption = random.choice(CORRUPTION_TYPES)
    if corruption == "null_required" and not nullable:
        return None
    elif corruption == "out_of_range" and field_type == "DoubleType":
        return random.choice([999.99, -999.99])
    elif corruption == "missing_field" and field_type == "StringType":
        return ""
    elif corruption == "late_timestamp" and field_type == "TimestampType":
        days_late = random.randint(30, 90)
        return datetime.now() - timedelta(days=days_late)
    return None

def generate_records(schema_fields: list, domain: str, num_records: int, corrupt_pct: float) -> List[Dict]:
    """Generar N registros sintéticos."""
    config = DOMAIN_FIELD_CONFIG.get(domain, {})
    records = []
    num_corrupt = int(num_records * corrupt_pct)
    num_clean = num_records - num_corrupt
    
    for _ in range(num_clean):
        row = {}
        for field in schema_fields:
            value = generate_value(field["name"], field["type"], config, row)
            # IMPORTANTE: Campos NOT NULL nunca deben ser NULL
            if not field.get("nullable", True) and value is None:
                # Generar default según tipo
                if field["type"] == "StringType":
                    value = f"{field['name']}_default"
                elif field["type"] == "TimestampType":
                    value = datetime.now()
                elif field["type"] == "DateType":
                    value = datetime.now().date()
                elif field["type"] == "IntegerType":
                    value = 0
                elif field["type"] == "DoubleType":
                    value = 0.0
            row[field["name"]] = value
        records.append(row)
    
    for _ in range(num_corrupt):
        row = {}
        fields_to_corrupt = random.sample([f["name"] for f in schema_fields], k=min(random.randint(1, 2), len(schema_fields)))
        for field in schema_fields:
            if field["name"] in fields_to_corrupt and field.get("nullable", True):
                # Solo corromper si es nullable
                row[field["name"]] = corrupt_value(field["name"], field["type"], field.get("nullable", True))
            else:
                value = generate_value(field["name"], field["type"], config, row)
                # Proteger campos NOT NULL incluso en registros corruptos
                if not field.get("nullable", True) and value is None:
                    if field["type"] == "StringType":
                        value = f"{field['name']}_default"
                    elif field["type"] == "TimestampType":
                        value = datetime.now()
                    elif field["type"] == "DateType":
                        value = datetime.now().date()
                    elif field["type"] == "IntegerType":
                        value = 0
                    elif field["type"] == "DoubleType":
                        value = 0.0
                row[field["name"]] = value
        records.append(row)
    
    random.shuffle(records)
    return records

print("Funciones de generación de datos cargadas")

Funciones de generación de datos cargadas


In [0]:
# Celda 5: Generar datos continuamente durante min_duration minutos

from pyspark.sql.types import StructType, StructField

def build_spark_schema(schema_fields: list) -> StructType:
    """Construir StructType de PySpark."""
    fields = []
    for f in schema_fields:
        spark_type = TYPE_MAP.get(f["type"], StringType())
        fields.append(StructField(f["name"], spark_type, True))
    return StructType(fields)

spark_schema = build_spark_schema(schema_fields)

print(f"\nEsquema Spark preparado: {len(schema_fields)} campos")


Generando 1000 registros para 'iot_sensors'...
Registros generados: 950 limpios + 50 corruptos
DataFrame creado: 1000 registros, 8 columnas
Preparando serialización Avro para Event Hubs...


In [0]:
# Celda 6: Publicar/Escribir datos en loop durante min_duration minutos

import time

start_time = datetime.now()
end_time_target = start_time + timedelta(minutes=min_duration)

total_sent = 0
total_failed = 0
batch_count = 0
sleep_interval = 30  # segundos entre batches

# Configuración Avro (solo para IoT)
if domain == "iot_sensors":
    def build_avro_schema(schema_fields: list) -> dict:
        """Construir schema Avro desde YAML schema_fields."""
        type_map = {
            "StringType": "string",
            "DoubleType": "double",
            "IntegerType": "int",
            "TimestampType": "long",
            "DateType": "int",
        }
        
        fields = []
        for f in schema_fields:
            avro_type = type_map.get(f["type"], "string")
            field_schema = {
                "name": f["name"],
                "type": ["null", avro_type] if f.get("nullable", True) else avro_type
            }
            fields.append(field_schema)
        
        return {
            "type": "record",
            "name": "IoTSensorReading",
            "namespace": "com.farmia.iot",
            "fields": fields
        }
    
    import fastavro
    from io import BytesIO
    from datetime import date as date_type
    
    avro_schema = build_avro_schema(schema_fields)
    
    AVRO_CONVERTERS = {
        "StringType": lambda v: str(v) if v is not None else None,
        "DoubleType": lambda v: float(v) if v is not None else None,
        "IntegerType": lambda v: int(v) if v is not None else None,
        "TimestampType": lambda v: int(v.timestamp() * 1000) if isinstance(v, datetime) else None,
        "DateType": lambda v: (v - date_type(1970, 1, 1)).days if isinstance(v, date_type) and v is not None else None,
    }
    
    def record_to_avro(record: dict, schema_fields: list, batch_id: int) -> bytes:
        """Serializar a Avro bytes con validación de tipos."""
        enriched_record = {}
        
        for field in schema_fields:
            field_name = field["name"]
            field_type = field["type"]
            value = record.get(field_name)
            
            try:
                converter = AVRO_CONVERTERS.get(field_type, lambda v: v)
                converted_value = converter(value)
                enriched_record[field_name] = converted_value
            except Exception as e:
                if not field.get("nullable", True):
                    raise
                enriched_record[field_name] = None
        
        enriched_record["_batch_id"] = batch_id
        enriched_record["_ingest_timestamp"] = int(datetime.now().timestamp() * 1000)
        enriched_record["_source_system"] = "iot_sensors"
        
        output = BytesIO()
        fastavro.schemaless_writer(output, avro_schema, enriched_record)
        return output.getvalue()
    
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def send_batch_to_eventhub(producer, batch_records):
        batch = producer.create_batch()
        event_count = 0
        
        for partition_key, avro_bytes in batch_records:
            event = EventData(avro_bytes)
            event.properties = {
                "sensor_id": partition_key,
                "format": "avro",
                "timestamp": datetime.now().isoformat()
            }
            
            try:
                batch.add(event)
                event_count += 1
            except ValueError:
                producer.send_batch(batch)
                batch = producer.create_batch()
                batch.add(event)
                event_count += 1
        
        if event_count > 0:
            producer.send_batch(batch)
        
        return event_count

# ===== LOOP PRINCIPAL: Generar y enviar datos durante min_duration minutos =====

print(f"\nIniciando loop de {min_duration} minutos...")
print(f"Generando {num_records_per_batch} registros por lote con pausa de {sleep_interval}s\n")

loop_iteration = 0

while datetime.now() < end_time_target:
    loop_iteration += 1
    elapsed_minutes = (datetime.now() - start_time).total_seconds() / 60
    
    print(f"\n[Lote {loop_iteration}] Tiempo transcurrido: {elapsed_minutes:.1f}min")
    
    # Generar registros para este lote
    records = generate_records(schema_fields, domain, num_records_per_batch, corrupt_pct)
    num_clean = int(num_records_per_batch * (1 - corrupt_pct))
    num_corrupt = num_records_per_batch - num_clean
    
    print(f"Generados: {num_clean} limpios + {num_corrupt} corruptos")
    
    if domain == "iot_sensors":
        # IoT: Serializar y enviar a Event Hubs
        serialized_records = []
        failed_in_batch = 0
        
        for i, record in enumerate(records):
            try:
                avro_bytes = record_to_avro(record, schema_fields, batch_id=loop_iteration)
                sensor_id = record.get("sensor_id", "unknown")
                serialized_records.append((sensor_id, avro_bytes))
            except Exception as e:
                failed_in_batch += 1
        
        sent_in_batch = 0
        try:
            with EventHubProducerClient.from_connection_string(
                connection_str,
                eventhub_name=eh_name,
                logging_enable=False
            ) as producer:
                
                for i in range(0, len(serialized_records), batch_size):
                    sub_batch = serialized_records[i:i+batch_size]
                    
                    try:
                        sent = send_batch_to_eventhub(producer, sub_batch)
                        sent_in_batch += sent
                    except Exception as e:
                        print(f"Error enviando sub-batch: {e}")
        
        except Exception as e:
            print(f"Error conectando Event Hubs: {e}")
        
        total_sent += sent_in_batch
        total_failed += failed_in_batch
        
        print(f"  📤 Enviados: {sent_in_batch}/{len(records)} registros a Event Hubs")
        if failed_in_batch > 0:
            print(f"  ⚠️  Fallidos: {failed_in_batch} registros")
    
    else:
        # Otros dominios: Escribir a Blob Storage
        df_batch = spark.createDataFrame(records, schema=spark_schema)
        
        writer = df_batch.write.format(source_format).mode("append")
        
        valid_partitions = [p for p in partition_by if p in df_batch.columns]
        if valid_partitions:
            writer = writer.partitionBy(*valid_partitions)
        
        try:
            writer.save(source_path)
            sent_in_batch = len(records)
            print(f"Escritos: {sent_in_batch} registros en Blob Storage")
            total_sent += sent_in_batch
        except Exception as e:
            print(f"Error escribiendo: {e}")
            total_failed += len(records)
    
    # Verificar si debe continuar
    remaining_time = (end_time_target - datetime.now()).total_seconds() / 60
    
    if remaining_time > 0:
        print(f"Tiempo restante: {remaining_time:.1f}min → Esperando {sleep_interval}s antes del próximo lote...")
        time.sleep(sleep_interval)
    else:
        print(f"Tiempo límite alcanzado")
        break

print("\n" + "="*70)
print("EJECUCIÓN COMPLETADA")
print("="*70)

end_execution_time = datetime.now()
total_duration_seconds = (end_execution_time - start_time).total_seconds()
total_duration_minutes = total_duration_seconds / 60
throughput = total_sent / total_duration_seconds if total_duration_seconds > 0 else 0


Serializando a Avro para Event Hubs...
Error serializando registro 55: must be string on field location_id
Error serializando registro 155: an integer is required on field reading_date
Error serializando registro 244: an integer is required on field reading_timestamp
Error serializando registro 336: must be string on field sensor_id
Error serializando registro 391: an integer is required on field reading_date
... y 25 errores más
Registros serializados: 970/1000

Publicando en Event Hubs (batch_size=100)...
Batch 1: 100 registros enviados
Batch 2: 100 registros enviados
Batch 3: 100 registros enviados
Batch 4: 100 registros enviados
Batch 5: 100 registros enviados
Batch 6: 100 registros enviados
Batch 7: 100 registros enviados
Batch 8: 100 registros enviados
Batch 9: 100 registros enviados
Batch 10: 70 registros enviados


In [0]:
# Celda 7: Resumen final con métricas de ejecución continua

print("\n" + "="*70)
print("RESUMEN — DATA GENERATOR FARMIA")
print("="*70)

summary_data = {
    "Dominio": domain,
    "Modo": "Ejecución continua (loop temporal)",
    "Duración objetivo": f"{min_duration} minutos",
    "Duración real": f"{total_duration_minutes:.2f} minutos",
    "Registros por lote": num_records_per_batch,
    "Total de lotes": loop_iteration,
    "Total registros enviados": total_sent,
    "Total registros fallidos": total_failed,
    "Tasa éxito": f"{(total_sent/(total_sent+total_failed))*100:.1f}%" if (total_sent+total_failed) > 0 else "N/A",
    "Throughput": f"{throughput:.1f} reg/s",
    "Intervalo entre lotes": f"{sleep_interval}s",
}

if domain == "iot_sensors":
    summary_data["Destino"] = f"Event Hubs: {eh_namespace}/{eh_name}"
    summary_data["Formato"] = "Avro"
else:
    summary_data["Destino"] = f"Blob Storage: {source_path}"
    summary_data["Formato"] = source_format

for key, value in summary_data.items():
    print(f"  {key:.<30} {value}")

print("="*70)

# Validación
if total_failed == 0 and total_sent > 0:
    print(f"\nÉXITO: {loop_iteration} lotes completados sin errores")
    print(f"   {total_sent} registros procesados exitosamente")
    if domain == "iot_sensors":
        print(f"   Datos disponibles en Event Hubs para consumo continuo")
    else:
        print(f"   Datos disponibles en Blob Storage para Auto Loader")
else:
    print(f"\nPARCIAL: {total_sent} registros enviados, {total_failed} fallidos")
    print(f"   Revisar los errores en los logs anteriores")


RESUMEN — DATA GENERATOR FARMIA
  Dominio.................. iot_sensors
  Registros generados...... 1000
  Registros limpios........ 950
  Registros corruptos...... 50
  % Corruptos.............. 5.0%
  Registros procesados..... 970
  Registros fallidos....... 30
  Tasa de éxito............ 97.0%
  Timestamp inicio......... 2026-05-19T06:36:49.545644
  Timestamp fin............ 2026-05-19T06:36:50.356004
  Duración................. 0.81s
  Throughput............... 1197.0 reg/s
  Destino.................. Event Hubs: stfarmia-ns/telemetry
  Formato.................. Avro

PARCIAL: 970/1000 registros procesados.
   Revisar los 30 registros fallidos.


In [ ]:
# Celda 8: VALIDACIÓN — Verificar que los datos se escribieron correctamente

print("\n" + "="*70)
print("VALIDACIÓN DE DATOS ESCRITOS — VERIFICAR TABLAS")
print("="*70)

from pyspark.sql.types import StructType

# Verificar tablas según el dominio
if domain == "iot_sensors":
    silver_table = "silver.iot_sensors"
    bronze_table = "bronze.iot_sensors"
elif domain == "weather_external":
    silver_table = "silver.weather_external"
    bronze_table = "bronze.weather_external"
elif domain == "sales_online":
    silver_table = "silver.sales_online"
    bronze_table = "bronze.sales_online"
else:  # inventory_erp
    silver_table = "silver.inventory_erp"
    bronze_table = "bronze.inventory_erp"

validation_results = {}

try:
    # Verificar existencia y contar filas en Silver
    silver_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {silver_table}").collect()[0]['cnt']
    silver_schema = spark.table(silver_table).schema
    validation_results["Silver"] = {
        "exists": True,
        "row_count": silver_count,
        "columns": len(silver_schema.fields),
        "schema_fields": [f.name for f in silver_schema.fields]
    }
    
    # Verificar sample row
    sample = spark.sql(f"SELECT * FROM {silver_table} LIMIT 1").collect()
    if sample:
        sample_data = sample[0].asDict()
        validation_results["Silver"]["sample_keys"] = list(sample_data.keys())
        validation_results["Silver"]["_event_timestamp_null"] = sample_data.get("_event_timestamp") is None
    
except Exception as e:
    validation_results["Silver"] = {
        "exists": False,
        "error": str(e)
    }

try:
    # Verificar existencia y contar filas en Bronze
    bronze_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {bronze_table}").collect()[0]['cnt']
    validation_results["Bronze"] = {
        "exists": True,
        "row_count": bronze_count
    }
except Exception as e:
    validation_results["Bronze"] = {
        "exists": False,
        "error": str(e)
    }

# Imprimir resultados
for layer, info in validation_results.items():
    print(f"\n📊 {layer}:")
    for key, value in info.items():
        if key == "schema_fields":
            print(f"   {key}: {', '.join(value[:5])}{'...' if len(value) > 5 else ''}")
        elif key == "sample_keys":
            print(f"   {key}: {', '.join(value[:5])}{'...' if len(value) > 5 else ''}")
        else:
            print(f"   {key}: {value}")

# Verificar problemas críticos
print("\n" + "="*70)
print("DIAGNÓSTICO:")

if validation_results.get("Silver", {}).get("exists"):
    silver_rows = validation_results["Silver"]["row_count"]
    if silver_rows == 0:
        print(f"⚠️  ADVERTENCIA: Silver table empty (0 filas)")
    elif silver_rows > 0:
        print(f"✅ Silver table contains {silver_rows} rows")
    
    # Verificar NULL en _event_timestamp
    if validation_results["Silver"].get("_event_timestamp_null"):
        print(f"❌ ERROR: _event_timestamp is NULL in sample row!")
    else:
        print(f"✅ _event_timestamp is NOT NULL")
else:
    print(f"❌ ERROR: Silver table not found or not accessible")

if validation_results.get("Bronze", {}).get("exists"):
    bronze_rows = validation_results["Bronze"]["row_count"]
    print(f"ℹ️  Bronze table has {bronze_rows} rows (quarantine/late data)")
else:
    print(f"ℹ️  Bronze table not yet created or empty")

print("="*70)